In [ ]:

from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    MessagesPlaceholder,
    FewShotPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_ollama import ChatOllama

# PipelinePromptTemplate is a legacy API.
# Try legacy import paths; availability depends on your version.
try:
    from langchain_classic.prompts.pipeline import PipelinePromptTemplate
except ImportError:
    try:
        from langchain.prompts.pipeline import PipelinePromptTemplate
    except ImportError:
        PipelinePromptTemplate = None



# Shared local Qwen model; each section invokes it separately.
model = ChatOllama(model="qwen2.5:1.5b", temperature=0, num_predict=10)


# ============================================================
# 1. PromptTemplate
# Creates a reusable text prompt.
# ============================================================

prompt1 = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

chain1 = prompt1 | model
response1 = chain1.invoke({"topic": "Java Streams"})
# Expected output: A simple explanation of Java Streams.

print("\n1. PromptTemplate")
print(response1.content)


# ============================================================
# 2. ChatPromptTemplate
# Combines system and human messages.
# ============================================================

prompt2 = ChatPromptTemplate.from_messages([ #python array
    ("system", "You are a Java interview tutor."), #python tuple ("key","value")
    ("human", "Explain {topic} simply.") #prompt inside a tuple
])

chain2 = prompt2 | model
response2 = chain2.invoke({"topic": "HashMap"})
# Expected output: A simple explanation of HashMap.

print("\n2. ChatPromptTemplate")
print(response2.content)


# ============================================================
# 3. SystemMessagePromptTemplate
# Creates a system message defining the model's role.
# ============================================================

prompt3 = SystemMessagePromptTemplate.from_template(
    "You are an experienced {role}."
)

system_message = prompt3.format(role="Java developer")

response3 = model.invoke([#python array
    ("human", "What is dependency injection?"), #passed a tuple 
    system_message
])
# Expected output: An explanation of dependency injection
# from the perspective of a Java developer.

print("\n3. SystemMessagePromptTemplate")
print(response3.content)


# ============================================================
# 4. HumanMessagePromptTemplate
# Creates a formatted user message.
# ============================================================

prompt4 = HumanMessagePromptTemplate.from_template(
    "Explain {topic} for a Java interview."
)

human_message = prompt4.format(topic="Microservices")

response4 = model.invoke([human_message])
# Expected output: An interview-oriented explanation
# of microservices.

print("\n4. HumanMessagePromptTemplate")
print(response4.content)


# ============================================================
# 5. AIMessagePromptTemplate
# Creates an assistant message, useful for conversation
# examples or providing previous assistant responses.
# ============================================================

prompt5 = AIMessagePromptTemplate.from_template(
    "A HashMap stores key-value pairs."
)

ai_message = prompt5.format()

response5 = model.invoke([
    ("system", "You are a Java tutor."),
    ("human", "What is a HashMap?"),
    ai_message,
    ("human", "Give me one real-world use.")
])
# Expected output: One practical use of HashMap.

print("\n5. AIMessagePromptTemplate")
print(response5.content)


# ============================================================
# 6. MessagesPlaceholder
# Inserts previous messages into the chat prompt.
# ============================================================

prompt6 = ChatPromptTemplate.from_messages([
    ("system", "You are a Java tutor. Little insane"),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain6 = prompt6 | model

response6 = chain6.invoke({
    "history": [
        ("human", "What is a HashMap?"),
        ("ai", "It is a map made of hashes used to find direction")
    ],
    "question": "What happens when two keys have the same hash?"
})
# Expected output: An explanation of hash collisions
# and how HashMap handles them.

print("\n6. MessagesPlaceholder")
print(response6.content)


# ============================================================
# 7. FewShotPromptTemplate
# Includes examples in a text prompt to guide the answer.
# ============================================================

examples = [
    {"question": "What is JDK?",
     "answer": "Tools for Java development."},
    {"question": "What is JVM?",
     "answer": "Runs Java bytecode."}
]

example_template = PromptTemplate.from_template(
    "Question: {question}\nAnswer: {answer}"
)

prompt7 = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Answer Java questions briefly using these examples:",
    suffix="Question: {question}\nAnswer:",
    input_variables=["question"]
)

chain7 = prompt7 | model

response7 = chain7.invoke({"question": "What is KGF?"})
# Expected output: An explanation of the Java Runtime
# Environment, following the supplied examples.

print("\n7. FewShotPromptTemplate")
print(response7.content)


# ============================================================
# 8. FewShotChatMessagePromptTemplate
# Adds example conversations to a chat prompt.
# ============================================================

chat_examples = [
    {"question": "What is JDK?",
     "answer": "Java development tools."},
    {"question": "What is JVM?",
     "answer": "Executes Java bytecode."}
]

example_chat = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])

few_shot_chat = FewShotChatMessagePromptTemplate(
    examples=chat_examples,
    example_prompt=example_chat
)

prompt8 = ChatPromptTemplate.from_messages([
    ("system", "You are a concise Java interview tutor."),
    few_shot_chat,
    ("human", "{question}")
])

chain8 = prompt8 | model

response8 = chain8.invoke({"question": "What is JRE?"})
# Expected output: A concise JRE explanation influenced
# by the example question-answer pairs.

print("\n8. FewShotChatMessagePromptTemplate")
print(response8.content)


# ============================================================
# 9. PipelinePromptTemplate (LEGACY API)
# Combines smaller prompt templates into one final prompt.
# This is prompt composition, not multiple LLM invocations.
# ============================================================

if PipelinePromptTemplate is not None:

    introduction_prompt = PromptTemplate.from_template(
        "You are a Java interview tutor."
    )

    topic_prompt = PromptTemplate.from_template(
        "Topic: {topic}"
    )

    question_prompt = PromptTemplate.from_template(
        "Explain the topic in simple words: {topic}"
    )

    # The output of each sub-template becomes a named
    # variable that the final prompt can reuse.
    final_prompt = PromptTemplate.from_template(
        "{introduction}\n{topic_details}\n{question}"
    )

    pipeline_prompt = PipelinePromptTemplate(
        final_prompt=final_prompt,
        pipeline_prompts=[
            ("introduction", introduction_prompt),
            ("topic_details", topic_prompt),
            ("question", question_prompt)
        ]
    )

    chain9 = pipeline_prompt | model

    response9 = chain9.invoke({"topic": "Java Streams"})
    # Expected output: A simple explanation of Java Streams
    # using the composed introduction and topic instructions.

    print("\n9. PipelinePromptTemplate")
    print(response9.content)

else:
    print("\n9. PipelinePromptTemplate")
    print("Legacy class is unavailable in this environment.")
    print("Use modern prompt composition with the | operator.")



1. PromptTemplate
Java Streams is a powerful feature in Java that allows

2. ChatPromptTemplate
HashMap is a data structure in Java that implements the

3. SystemMessagePromptTemplate
Dependency injection is a design pattern in software engineering that

4. HumanMessagePromptTemplate
Microservices is a software architecture style that involves breaking

5. AIMessagePromptTemplate
A HashMap is often used to store and retrieve data

6. MessagesPlaceholder
When two keys have the same hash, a collision

7. FewShotPromptTemplate
Answer: A type of Java library.

8. FewShotChatMessagePromptTemplate
Java Runtime Environment.

9. PipelinePromptTemplate
Legacy class is unavailable in this environment.
Use modern prompt composition with the | operator.
